# PlotlyBrain — workflow 
### (QUINT output → scores → recolored Allen SVGs)

In [ ]:
import os
from plotlybrain.scores import save_scores
from plotlybrain.plotly_render import load_score

### Compute a score table from QUINT exports
You can compute either *frequency* (fraction of animals with objects > 0 in each region), or *rel_abundance* (z-scored total object counts across regions)

In [ ]:
DATA_DIR = "/Users/annateruel/Desktop/cfos_mecp2_maria/all_double"  # <-- EDIT
OUT_DIR  = "/Users/annateruel/Desktop/cfos_mecp2_maria/all_double/output/"   # <-- EDIT
META_PATH = "/Users/annateruel/Desktop/cfos_mecp2_maria/all_double/metadata_double.csv" # <-- EDIT
os.makedirs(OUT_DIR, exist_ok=True)
print("DATA_DIR:", DATA_DIR)
print("OUT_DIR :", OUT_DIR)

Getting a df with the cell counts per every brain region

In [ ]:
from plotlybrain.scores import load_refatlas_regions, compute_animal_region_counts
import pandas as pd
metadata = pd.read_csv(META_PATH, sep=";")

df = load_refatlas_regions(
    data_dir=DATA_DIR,
    sep=";"
)

region_by_subject = compute_animal_region_counts(df)

region_by_subject = region_by_subject.merge(
    metadata,
    on="animal",
    how="left"
)

region_by_subject.to_csv(
    os.path.join('/Users/annateruel/Desktop/', "animal_region_counts_double_metadata.csv"),
    index=False
)

Getting the score table

In [ ]:
SCORE = "frequency"
csv_path = os.path.join(OUT_DIR, f"scores_{SCORE}.csv")
df_scores = save_scores(
    data_dir=DATA_DIR,
    metadata_path=META_PATH,
    out_path=OUT_DIR,
    metadata_sep=";",
    group_col=['genotype', 'sex', 'group'],
    score="rel_abundance",
    rel_abundance_method="reference",
    reference_mode="pooled",
)

### Load score lookups 

If you have already saved the scores, you can directly load them. 


In [ ]:
VALUE_COL = "relative_abundance_z"
csv_path = '/Users/annateruel/Desktop/cfos_mecp2_maria/all_cFos/output/rel_abundance_mecp2_female_caps.csv'
id2value, id2name, score_col = load_score(
    csv_path,
    id_col="Region ID",
    name_col="Region name",
    value_col=VALUE_COL,
)

### Converting Bregma Level to Slice Index

In [ ]:
from plotlybrain.coord_system import ap_mm_to_slice_index, slice_index_to_ap_mm

ap_mm = -2.0
slice_index = ap_mm_to_slice_index(ap_mm, resolution_um=25)

print("slice_index:", slice_index)
print("back-converted AP:", slice_index_to_ap_mm(slice_index, resolution_um=25))

### Convert Allen 3D annotations to Geo-JSON files

In [ ]:
from plotlybrain.build_polygons import (
    load_annotation_volume,
    load_structure_graph,
    get_slice_view,
    build_slice_geojson,
)

OUT_DIR = "/Users/annateruel/Desktop/allen_cache"

volume, header = load_annotation_volume(
    resolution_um=25,
    storage_mode="disk",
    cache_dir=OUT_DIR,
    overwrite=False,
)

structure_df = load_structure_graph(
    storage_mode="disk",
    cache_dir=OUT_DIR,
    overwrite=False,
)
slice_img = get_slice_view(volume, slice_index, "coronal")
gj = build_slice_geojson(
    slice_img=slice_img,
    structure_df=structure_df,
    slice_index=slice_index,
    orientation="coronal",
    resolution_um=25,
    min_area_px=5,
    simplify_px=0.8,
)

### Plotly Geo-JSON render

Build df from the GeoJSON slice

In [ ]:
import pandas as pd
df = pd.DataFrame([f["properties"] for f in gj["features"]])
df["score"] = df["Region ID"].map(id2value)
df["Region name"] = df["Region ID"].map(id2name)

In [ ]:
from plotlybrain.plotly_render import render_brain_slice

id2value, id2name, score_col = load_score(
    csv_path,
    value_col="relative_abundance_z",
)

fig = render_brain_slice(
    geojson_obj=gj,
    id2value=id2value,
    id2name=id2name,
    title="Brain slice test",
    score_name=score_col,
    line_color="white",
    line_width=0.8,
    line_shape="spline",
    smoothing=0.7,
    show_colorbar=True,
)

fig.show()